# CartPole-v1

In [ ]:
import logging
from pathlib import Path

import gymnasium as gym
import mlflow

from engram.dreamerv3.agent import DreamerV3
from engram.tools.conditionals import Every

# resolve project root from notebook's parent dir
project_root = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{project_root / 'mlflow.db'}")
mlflow.set_experiment("dreamerv3-cartpole")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("cartpole")

## Hyperparameters

In [ ]:
params = {
    # rssm
    "n_categoricals": 8,
    "n_classes": 8,
    "encoder_output_size": 64,
    "recurrent_size": 128,
    "posterior_hidden_sizes": [128],
    "prior_hidden_sizes": [128],
    "posterior_activation": "rmsnorm+silu",
    "prior_activation": "rmsnorm+silu",
    "unimix": 0.01,
    # heads
    "reward_hidden_sizes": [128],
    "reward_activation": "rmsnorm+silu",
    "continue_hidden_sizes": [128],
    "continue_activation": "rmsnorm+silu",
    # actor + critic
    "actor_hidden_sizes": [128],
    "actor_activation": "rmsnorm+silu",
    "critic_hidden_sizes": [128],
    "critic_activation": "rmsnorm+silu",
    # encoder + decoder
    "encoder_hidden_sizes": [128],
    "encoder_activation": "rmsnorm+silu",
    "decoder_hidden_sizes": [128],
    "decoder_activation": "rmsnorm+silu",
    # two-hot
    "two_hot_low": -20.0,
    "two_hot_high": 20.0,
    "two_hot_n_bins": 41,
    # optim
    "world_model_lr": 3e-4,
    "actor_lr": 3e-5,
    "critic_lr": 3e-5,
    "critic_tau": 0.02,
}

train_config = {
    "steps": 25_000,
    "batch_size": 16,
    "sequence_length": 64,
    "prefill_steps": 1024,
    "device": "cpu",
}

## Agent

In [ ]:
env = gym.make("CartPole-v1")

agent = DreamerV3(
    env=env,
    params=params,
    device=train_config["device"],
    batch_size=train_config["batch_size"],
    sequence_length=train_config["sequence_length"],
    prefill_steps=train_config["prefill_steps"],
)

## Train

Run training and stream metrics to MLflow. The generator yields once per env step; we log every metric in the dict against the env step as the x-axis.

In [ ]:
should_eval = Every(n=2_500)
should_log = Every(n=1_000)
should_save = Every(n=10_000)

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_params(train_config)

    # train loop
    for m in agent.train(steps=train_config["steps"]):
        step: int = m.pop("step")  # type: ignore
        gradient_step: int = m.pop("gradient_step")  # type: ignore
        for k, v in m.items():
            mlflow.log_metric(k, v, step=step)

        if should_eval():
            eval_metrics = agent.eval(n_episodes=10)
            for k, v in eval_metrics.items():
                mlflow.log_metric(k, v, step=step)
            logger.info("step %d eval: %s", step, eval_metrics)

        if should_log():
            logger.info("step %d: %s", step, m)

        if should_save():
            agent.save(
                "cartpole_env{step:07d}_grad{gradient_step:07d}",
                env_step=step,
                gradient_step=gradient_step,
            )

    # final eval
    final_eval = agent.eval(n_episodes=100)
    for k, v in final_eval.items():
        mlflow.log_metric(f"final_{k}", v, step=train_config["steps"])
    logger.info("final eval: %s", final_eval)

In [ ]:
agent.save(
    "cartpole_solved.pt",
    env_step=step,
    gradient_step=gradient_step,
)